In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "sumy", "bert-score", "sacrebleu"], check=True)

import os
import math
import numpy as np
import pandas as pd
import torch
import chardet
import evaluate
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # для Colab без GUI

# sumy — для экстрактивных методов
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer   # PageRank на графе предложений
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer as SumyStemmer
from sumy.utils import get_stop_words

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/ruT5_science_sum_gazeta"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "overfit_checkpoints")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "overfit_final_model")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

model_checkpoint = "IlyaGusev/rut5_base_sum_gazeta"

max_input_length = 600
max_target_length = 200

device = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
print("\n" + "="*50)
print("Загрузка IIS датасета...")
os.system("rm -rf /content/summarization-dataset")
os.system("git clone https://github.com/iis-research-team/summarization-dataset.git /content/summarization-dataset")

texts_iis, abstracts_iis = [], []
data_path = "/content/summarization-dataset/dataset"

for domain in os.listdir(data_path):
    domain_path = os.path.join(data_path, domain)
    if os.path.isdir(domain_path):
        for paper_folder in os.listdir(domain_path):
            paper_path = os.path.join(domain_path, paper_folder)
            text_file = os.path.join(paper_path, "text.txt")
            abstract_file = os.path.join(paper_path, "abstract.txt")

            if os.path.exists(text_file) and os.path.exists(abstract_file):
                with open(text_file, "r", encoding="utf-8") as f:
                    text = f.read().strip()
                with open(abstract_file, "r", encoding="utf-8") as f:
                    abstract = f.read().strip()

                if text and abstract:
                    texts_iis.append(text)
                    abstracts_iis.append(abstract)

print(f"IIS: загружено {len(texts_iis)} пар")

In [ ]:
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)
        result = chardet.detect(raw_data)
        return result['encoding'] or 'utf-8'

def read_file_with_encoding(file_path):
    encoding = detect_encoding(file_path)
    try:
        with open(file_path, 'r', encoding=encoding) as f:
            return f.read()
    except Exception:
        for enc in ['utf-8', 'windows-1251', 'cp1251', 'koi8-r', 'iso-8859-5']:
            try:
                with open(file_path, 'r', encoding=enc) as f:
                    return f.read()
            except Exception:
                continue
        raise UnicodeDecodeError(f"Не удалось прочитать файл {file_path}")

print("\n" + "="*50)
print("Загрузка RuSciTextSum...")
os.system("rm -rf /content/RuSciTextSum")
os.system("git clone https://github.com/Svetych/RuSciTextSum.git /content/RuSciTextSum")

# --- RuSciText ---
texts_rusci, abstracts_rusci = [], []
rusci_path = "/content/RuSciTextSum/dataset/ruscitext"

if os.path.exists(rusci_path):
    for filename in os.listdir(rusci_path):
        file_path = os.path.join(rusci_path, filename)
        if os.path.isfile(file_path):
            try:
                content = read_file_with_encoding(file_path)
                lines = content.strip().split('\n\n')
                if len(lines) >= 2:
                    text = '\n\n'.join(lines[:-1]).strip()
                    abstract = lines[-1].strip()
                    if text and abstract:
                        texts_rusci.append(text)
                        abstracts_rusci.append(abstract)
            except Exception as e:
                print(f"  Ошибка при чтении {filename}: {e}")
else:
    print("⚠️ Путь ruscitext не найден:", rusci_path)

print(f"RuSciText: загружено {len(texts_rusci)} пар")

# --- RuArchive ---
texts_archive, abstracts_archive = [], []
archive_path = "/content/RuSciTextSum/dataset/ruarchive"

if os.path.exists(archive_path):
    for filename in os.listdir(archive_path):
        file_path = os.path.join(archive_path, filename)
        if os.path.isfile(file_path):
            try:
                content = read_file_with_encoding(file_path)
                lines = content.strip().split('\n\n')
                if len(lines) >= 2:
                    text = lines[0].strip()
                    abstract = lines[1].strip()
                    if text and abstract:
                        texts_archive.append(text)
                        abstracts_archive.append(abstract)
            except Exception as e:
                print(f"  Ошибка при чтении {filename}: {e}")
else:
    print("⚠️ Путь ruarchive не найден:", archive_path)

print(f"RuArchive: загружено {len(texts_archive)} пар")


In [ ]:
print("\n" + "="*50)
print("Загрузка 200 примеров из Газеты...")
gazeta_raw = load_dataset("IlyaGusev/gazeta", split="train")
gazeta_sample = gazeta_raw.shuffle(seed=42).select(range(200))
texts_gazeta = list(gazeta_sample["text"])
abstracts_gazeta = list(gazeta_sample["summary"])
print(f"Газета: загружено {len(texts_gazeta)} пар")

In [ ]:
print("\n" + "="*50)
print("Объединение датасетов...")

parts = []
if texts_iis:
    parts.append(Dataset.from_dict({"text": texts_iis, "summary": abstracts_iis}))
    print(f"  + IIS:      {len(texts_iis)}")
if texts_rusci:
    parts.append(Dataset.from_dict({"text": texts_rusci, "summary": abstracts_rusci}))
    print(f"  + RuSciText: {len(texts_rusci)}")
if texts_archive:
    parts.append(Dataset.from_dict({"text": texts_archive, "summary": abstracts_archive}))
    print(f"  + RuArchive: {len(texts_archive)}")
if texts_gazeta:
    parts.append(Dataset.from_dict({"text": texts_gazeta, "summary": abstracts_gazeta}))
    print(f"  + Газета:   {len(texts_gazeta)}")

assert parts, "❌ Нет данных ни из одного источника!"

combined = concatenate_datasets(parts)
print(f"\nИтого примеров до разбивки: {len(combined)}")

split = combined.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"]
})

print(f"✅ Train:      {len(dataset['train'])} примеров")
print(f"✅ Validation: {len(dataset['validation'])} примеров")

In [ ]:
print("\n" + "="*50)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)
print("✅ Загружена модель:", model_checkpoint)

In [ ]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=max_input_length,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True,
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)
print(tokenized_datasets)

In [ ]:
small_train = tokenized_datasets["train"].select(range(32))
small_val = tokenized_datasets["validation"].select(range(8))

print("Train subset size:", len(small_train))
print("Val subset size:", len(small_val))

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [" ".join(x.strip().split()) for x in decoded_preds]
    decoded_labels = [" ".join(x.strip().split()) for x in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False
    )
    return {
        "rouge1": round(result["rouge1"], 4),
        "rouge2": round(result["rouge2"], 4),
        "rougeL": round(result["rougeL"], 4),
    }

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

In [ ]:
print("\n" + "="*50)
model.eval()

batch = data_collator([small_train[0], small_train[1]])
batch = {k: v.to(model.device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print("=== MANUAL LOSS CHECK BEFORE TRAIN ===")
print("Loss:", outputs.loss.item())
print("Finite:", torch.isfinite(outputs.loss).item())
print("Logits shape:", outputs.logits.shape)

labels_batch = batch["labels"]
print("=== LABELS CHECK ===")
print("Labels shape:", labels_batch.shape)
print("Count != -100:", (labels_batch != -100).sum().item())

valid = labels_batch[labels_batch != -100]
if len(valid) > 0:
    print("Min valid label:", valid.min().item())
    print("Max valid label:", valid.max().item())
    print("First 30 valid labels:", valid[:30].tolist())
else:
    print("❌ No valid labels found")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    num_train_epochs=8,
    weight_decay=0.01,

    warmup_steps=10,
    lr_scheduler_type="linear",

    predict_with_generate=True,
    generation_max_length=max_target_length,
    generation_num_beams=5,

    logging_steps=5,
    report_to="none",

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,

    fp16=False,
    remove_unused_columns=False,
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n" + "="*50)
print("Старт обучения...")
train_result = trainer.train()

print("\n=== TRAIN RESULT ===")
print(train_result)

print("\n=== TRAINER STATE ===")
print("Global step:", trainer.state.global_step)
print("Best metric:", trainer.state.best_metric)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Train epochs:", trainer.state.epoch)
print("Is local process zero:", trainer.is_world_process_zero())

print("\n=== LAST 20 LOG RECORDS ===")
for row in trainer.state.log_history[-20:]:
    print(row)

In [ ]:
# =========================
# 13. СОХРАНЕНИЕ МОДЕЛИ
# =========================
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("✅ Сохранена финальная модель в:", FINAL_MODEL_DIR)

In [ ]:
best_ckpt = trainer.state.best_model_checkpoint
print("\nBest checkpoint from trainer:", best_ckpt)

if best_ckpt is None:
    print("⚠️ best_model_checkpoint is None, fallback to FINAL_MODEL_DIR")
    best_ckpt = FINAL_MODEL_DIR

print("Checkpoint exists:", os.path.exists(best_ckpt))

finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt).to(device)
finetuned_tokenizer = AutoTokenizer.from_pretrained(best_ckpt)
print("✅ Загружен finetuned checkpoint:", best_ckpt)

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)
base_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
print("✅ Загружена базовая модель")